In [ ]:
import sys
import os
# Tell Python to look two folders up (the project root) so it can find 'src'
sys.path.append(os.path.abspath('../../'))

import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from src.utils.paths import PROCESSED_DATA_DIR
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

# Baseline KNN

In [ ]:
# Load the clean, scaled data your preprocessing script created
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").values.ravel()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").values.ravel()
# Initialize the KNN classifier with k=3
knn_estimator = KNeighborsClassifier(n_neighbors=3)

# Train the model on the training data
knn_estimator.fit(X_train, y_train)

print("KNN Model trained successfully!")
# Use the trained model to predict the test data
y_pred = knn_estimator.predict(X_test)
y_prob = knn_estimator.predict_proba(X_test)[:, 1] # Probabilities for ROC AUC

# Evaluate the results
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_prob))

## Baseline KNN Analysis

- The model uses `k=3`, which means predictions are based on the 3 nearest neighbors.
- This allows the model to capture local patterns, but also makes it sensitive to noise and class distribution.

- Accuracy: **0.965**
- ROC AUC: **0.957**

### Observations

- There is a noticeable class imbalance:
  - Extroverts (class 0): 4115 samples
  - Introverts (class 1): 1443 samples

- The model performs better on the majority class (extroverts), indicating a mild bias introduced by class imbalance.

- Lower recall for introverts (0.92) suggests that the model misses a portion of this class.

- The difference between macro and weighted averages indicates that overall performance is influenced by class imbalance.

### Model Behavior (KNN-specific)

- Using a small number of neighbors (`k=3`):
  - Helps capture local structure effectively
  - Can lead to sensitivity to noise and local density variations

- The slightly weaker performance on the minority class suggests that:
  - The model is influenced by the higher density of majority class samples
  - Minority class instances may be surrounded by majority neighbors

- KNN relies on distance-based similarity, making feature scaling essential for meaningful distance calculations.

### Takeaway

- The KNN model achieves strong overall performance, comparable to the decision tree.
- It demonstrates excellent class separation (high ROC AUC).
- However, there is a mild bias toward the majority class (extroverts), reflected in lower recall for introverts.
- Performance is robust, but could potentially be improved by tuning `k` or addressing class imbalance.

In [ ]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### Classification Report Insights

- Class 0 (Extrovert):
  - Precision: 0.97
  - Recall: 0.98
  - F1-score: 0.98
  - The model performs extremely well on this class, correctly identifying most extroverts.

- Class 1 (Introvert):
  - Precision: 0.94
  - Recall: 0.92
  - F1-score: 0.93
  - Performance is slightly lower, indicating that some introverts are misclassified as extroverts.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Extrovert", "Introvert"],
    cmap="Blues"
)

plt.title("KNN Confusion Matrix")
plt.show()

### Confusion Matrix Interpretation

- Most extroverts (class 0) are correctly classified, confirming high recall (0.98).
- Some introverts (class 1) are misclassified as extroverts, consistent with lower recall (0.92).
- This indicates a mild bias toward the majority class.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("KNN ROC Curve")
plt.show()

### ROC Curve Interpretation

- The ROC curve is close to the top-left corner, indicating strong classification performance.
- The high ROC AUC (~0.957) confirms that the model effectively separates the two classes.

In [ ]:
print("Train accuracy:", knn_estimator.score(X_train, y_train))
print("Test accuracy:", knn_estimator.score(X_test, y_test))

## Generalization Analysis

- Train accuracy: **0.970**
- Test accuracy: **0.965**

- The difference between train and test performance is very small (~0.005), indicating good generalization.

- There are no strong signs of overfitting:
  - The model performs consistently on unseen data.
  - It is not excessively memorizing the training set.
- The use of a small number of neighbors (`k=3`) does not lead to significant overfitting in this case, suggesting the data has well-defined local structure.

## Limitations

- KNN is computationally expensive at prediction time, as it computes distances to all training samples.
- The model is sensitive to the choice of `k` and local data distribution.
- Performance may degrade in high-dimensional spaces due to the curse of dimensionality.
- Class imbalance can influence predictions, leading to slightly worse performance on minority classes.

## Final Conclusion

- The KNN model achieves strong performance, with high accuracy (~0.965) and ROC AUC (~0.957).
- The model performs slightly better on the majority class, indicating a mild imbalance effect.
- Overall, the model generalizes well and captures meaningful patterns in the data.
- Performance is robust, but could be further improved by tuning `k` or addressing class imbalance.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

# 1. Define the parameter grid to search through
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 2. Set up the Inner and Outer loops
# We use StratifiedKFold because your classes are imbalanced (more Introverts than Extroverts)
cv_inner = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=42)
cv_outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# 3. Define the base model
knn = KNeighborsClassifier()

# 4. INNER LOOP: RandomizedSearchCV
search = RandomizedSearchCV(
    knn,
    param_distributions=param_grid,
    n_iter=15,
    scoring='accuracy',
    cv=cv_inner,
    random_state=42,
    n_jobs=-1
)

# 5. OUTER LOOP: cross_val_score evaluates the true performance
# Note: We use X_train and y_train here. The outer loop will split this further into Train/Test folds!
nested_scores = cross_val_score(search, X_train, y_train, scoring='accuracy', cv=cv_outer, n_jobs=-1)
print("Nested CV Accuracy Scores for each fold:", nested_scores)
print("True Expected Accuracy:", nested_scores.mean())

# 6. Finally, train the best model on the FULL training set to see what parameters won
search.fit(X_train, y_train)
print("\nBest Hyperparameters Found:", search.best_params_)